# Research Question
We investigate whether short-term predictability in corn returns, driven by cross-asset and market regime signals, can be exploited to improve hedging and trading decisions in agricultural commodity markets.

# Who Benefits Economically?
Farmers, traders, co-ops, merchandisers all hedge using futures. The timing for buying futures for hedging matters since if the expected returns are negative over the next few days, participants could wait for better execution prices instead of buying futures immediately which reduces hedging costs.

# Data Collection
Creating a dataset for corn future prices and supporting features.

## Feature Set:

- $\text{Cross Asset Features}$:
    - $\text{Corn Futures Returns}_{t+1} = Y$
    - $\text{Corn Futures Returns}_{t}$
    - $\text{Crude Oil Futures Rolling Z-score}_{t}$: Calculated on price level (not returns) to avoid the artificial 0% returns created in the data by f-filling
    - $\text{Soy Futures Rolling Z-score}_{t}$: Calculated on price level (not returns) to avoid the artificial 0% returns created in the data by f-filling
    - $\text{Corn-Soy Ratio}_{t}$
    - $\text{Corn-Oil Ratio}_{t}$
- $\text{Volatility/Regime Variables}$:
    - $\text{Rolling 5-day Realized Volatility (RV)}_{corn, t}$
    - $\text{Rolling 5-day Realized Volatility (RV)}_{corn, t-1}$
    - $\text{Rolling 5-day Realized Volatility (RV)}_{corn, t-2}$
    - $\text{Rolling 20-day RV}_{corn, t}$
	- $\text{Rolling Volume Z-score}_{corn, t}$
- $\text{Seasonality Features}$:
    - $\text{Month number}_{t}\isin[1,12]$
- $\text{Interest Rate Features}$:
    - $\text{Term Spread}_{t} = y_{10y} - y_{2y}$
    - $\text{Short-end Spread}_{t} = y_{2y} - y_{3m}$
    - $\text{Yield Curvature}_{t} = 2y_{5y} - y_{2y} - y_{10y}$
    - $\Delta\text{Term Spread}_{t} = \text{Term Spread}_{t} - \text{Term Spread}_{t-1}$
    - $\Delta\text{Term Spread}_{t-1} = \text{Term Spread}_{t-1} - \text{Term Spread}_{t-2}$
    - $\Delta\text{Term Spread}_{t-2} = \text{Term Spread}_{t-2} - \text{Term Spread}_{t-3}$
    - $\Delta\text{Short-end Spread}_{t} = \text{Short-end Spread}_{t} - \text{Short-end Spread}_{t-1}$
    - $\Delta\text{Yield Curvature}_{t} = \text{Yield Curvature}_{t} - \text{Yield Curvature}_{t-1}$
    - $\Delta\text{10-year Yield}_{t} = \text{10-year Yield}_{t} - \text{10-year Yield}_{t-1}$
    - $\Delta\text{2-year Yield}_{t} = \text{2-year Yield}_{t} - \text{2-year Yield}_{t-1}$
- $\text{Weather Features}$:
    - $\text{Temp}_{t}^{avg}$
    - $\text{Temp}_{t}^{min}$
    - $\text{Temp}_{t}^{max}$
    - $\text{Precipitation}_{t}^{avg}$
    - $\text{Relative Humidity}_{t}^{avg}$
    - $\text{Wind Speed}_{t}^{avg}$  
    - $\text{Heat Stress Proxy}_{t} = \text{Temp}_{t}^{avg} \cdot \text{Precipitation}_{t}^{avg}$  
**Note:** *All daily weather features are weighted averages of the different regional weather conditions in the Corn Belt region. The weights used are the production level (in bushels) of corn in the region.*
- $\text{Quantitative Features}$:
    - $\text{ATR}_{t}^{(14)} = \frac{1}{14}\sum_{i=0}^{13}\text{TR}_{t-i}$
    $\qquad\text{where }\text{TR}_{t}\text{=}\max\left\{H_t - L_t,\;|H_t - C_{t-1}|,\;|L_t - C_{t-1}|\right\}$
    $\text{and where H}_t,\text{ L}_t,\text{and C}_t\text{ denote the high, low, and close prices on day t}$
    - $\text{SMA Spread}_{t}^{(5, 20)} = \frac{\text{SMA}_{5}}{\text{SMA}_{20}} - 1$
    - $\text{Moving Average Convergence-Divergence (MACD)}_{t}^{(12, 26, 9)}$
    - $\text{Relative Strength Indicator (RSI)}_{t}^{(14)}$
    - $\text{RSI}_{t-1}^{(14)}$
    - $\text{RSI}_{t-2}^{(14)}$

In [1]:
import os
import ssl
import certifi

# Needed for meteostat
os.environ["SSL_CERT_FILE"] = certifi.where()
os.environ["REQUESTS_CA_BUNDLE"] = certifi.where()

ssl._create_default_https_context = lambda: ssl.create_default_context(
    cafile=certifi.where()
)

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import requests
import yfinance as yf
from pandas_datareader import data as pdr
from datetime import datetime
import meteostat as ms

# CROSS-ASSET DATA FETCH
print("Starting cross-asset data collection...")
corn = yf.download("ZC=F", start="2000-01-01", auto_adjust=True).droplevel(level=1,axis=1)[["Close","High","Low","Volume"]]
oil = yf.download("CL=F", start="2000-01-01", auto_adjust=True).droplevel(level=1,axis=1)[["Close","Volume"]]
soy = yf.download("ZS=F", start="2000-01-01", auto_adjust=True).droplevel(level=1,axis=1)[["Close","Volume"]]
rates = pdr.DataReader(["DGS10","DGS5","DGS2","DGS3MO"], "fred", start="2000-01-01").rename(columns={'DGS10':'T10_yield','DGS5':'T5_yield','DGS2':'T2_yield','DGS3MO':'T3M_yield'})

for name, cmdty in {'corn':corn, 'oil':oil, 'soy':soy}.items():
    cmdty.rename(columns={x:name+"_"+x.lower() for x in cmdty.columns}, inplace=True)
    cmdty.columns.name = None
    if name == 'corn':
        cmdty[name+'_ret'] = cmdty[name+'_close'].pct_change()

Starting cross-asset data collection...


[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


In [3]:
# WEATHER DATA FETCH

# Representative points (state capitals / central ag hubs)
start = datetime(2000, 1, 1)
end   = datetime(2026, 12, 31)
state_points = {
    "IA": ms.Point(41.5868, -93.6250),   # Des Moines
    "IL": ms.Point(40.6331, -89.3985),   # Springfield
    "NE": ms.Point(40.8136, -96.7026),   # Lincoln
    "IN": ms.Point(39.7684, -86.1581),   # Indianapolis
    "MN": ms.Point(44.9537, -93.0900),   # St Paul
    "OH": ms.Point(39.9612, -82.9988),   # Columbus
    "SD": ms.Point(44.3683, -100.3510),  # Pierre
}

def fetch_state_weather(state, point, start, end):
    stations = ms.stations.nearby(point, limit=1)
    df = ms.daily(stations, start, end).fetch()
    df["state"] = state
    print("State: "+state+"\tDONE")
    keep = ["time", "state", "temp", "tmin", "tmax", "prcp", "rhum", "wspd"]
    return df.reset_index()[keep].rename(columns={"time": "date"})

print("Starting weather data collection...")
wx = pd.concat([fetch_state_weather(state, point, start, end) for state, point in state_points.items()], ignore_index=True)
wx["date"] = pd.to_datetime(wx["date"])
wx = wx.sort_values(["state", "date"]).reset_index(drop=True)

# Open-source API so no privacy concerns
NASS_API_KEY = "39E23AA9-8DF0-3C0D-AA24-5B5CE725DD03"
def fetch_corn_production_weights(api_key, start_year, end_year):
    """
    Quick Stats query for state corn grain production.
    """

    if end_year >= 2026:
        print("End year too high...")
        return np.nan
    outputs = []
    url = "https://quickstats.nass.usda.gov/api/api_GET/"
    for year in range(start_year, end_year+1):
        params = {
            "key": api_key,
            "commodity_desc": "CORN",
            "statisticcat_desc": "PRODUCTION",
            "unit_desc": "BU",
            "agg_level_desc": "STATE",
            "year": year,
            "format": "JSON",
        }
        r = requests.get(url, params=params, timeout=30)
        r.raise_for_status()
        data = r.json()["data"]
        df = pd.DataFrame(data)

        # Keep state + value
        out = df[["state_alpha", "Value"]].copy()
        out["Value"] = (
            out["Value"]
            .str.strip()                      # remove leading/trailing spaces
            .str.replace(",", "", regex=False)
            .replace({"(D)": np.nan, "(Z)": np.nan, "": np.nan})
        )
        out["Value"] = pd.to_numeric(out["Value"], errors="coerce")
        out = out.dropna().rename(columns={"state_alpha": "state", "Value": "prod_bu"})

        # Restrict to the states we use and find avg prod for the year
        out = out[out["state"].isin(state_points.keys())].copy()
        out = out.groupby("state").mean().reset_index()
        out['year'] = year
        # Production share weights
        out["weight"] = out["prod_bu"] / out["prod_bu"].sum()
        print(f"Year: {year}\tDONE")
        outputs.append(out)

    out_df = pd.concat(outputs, axis=0)
    return out_df.sort_values("year")

print("\nStarting production weight collection...")
weights_df = fetch_corn_production_weights(NASS_API_KEY, start_year=start.year, end_year=end.year-1)
if len(weights_df[weights_df["year"] == 2026]) == 0:
    weights_2026 = weights_df[weights_df["year"] == 2025].copy()
    weights_2026["year"] = 2026
    weights_df = pd.concat([weights_df, weights_2026], ignore_index=True)

# Find weighted average of weather variables for each day using production amount as weights
wx["year"] = pd.to_datetime(wx['date']).dt.year
wx_w = wx.merge(weights_df[["state", "year", "weight"]], on=["state", "year"], how="left")
weather_cols = ["temp", "tmin", "tmax", "prcp", "rhum", "wspd"]
for col in weather_cols:
    wx_w[f"{col}_w"] = wx_w[col] * wx_w["weight"]
wx_w_dly = wx_w.groupby("date", as_index=False)[[f"{col}_w" for col in weather_cols]].sum().rename(columns={"temp_w": "prod_w_temp", "tmin_w": "prod_w_tmin", "tmax_w": "prod_w_tmax", "prcp_w": "prod_w_prcp","rhum_w": "prod_w_rhum","wspd_w": "prod_w_wspd"})
wx_w_dly['temp_prcp_interaction'] = wx_w_dly["prod_w_temp"] * wx_w_dly["prod_w_prcp"]   # Creating the interaction term for heat stress

Starting weather data collection...
State: IA	DONE
State: IL	DONE
State: NE	DONE
State: IN	DONE
State: MN	DONE
State: OH	DONE
State: SD	DONE

Starting production weight collection...
Year: 2000	DONE
Year: 2001	DONE
Year: 2002	DONE
Year: 2003	DONE
Year: 2004	DONE
Year: 2005	DONE
Year: 2006	DONE
Year: 2007	DONE
Year: 2008	DONE
Year: 2009	DONE
Year: 2010	DONE
Year: 2011	DONE
Year: 2012	DONE
Year: 2013	DONE
Year: 2014	DONE
Year: 2015	DONE
Year: 2016	DONE
Year: 2017	DONE
Year: 2018	DONE
Year: 2019	DONE
Year: 2020	DONE
Year: 2021	DONE
Year: 2022	DONE
Year: 2023	DONE
Year: 2024	DONE
Year: 2025	DONE


In [4]:
# Rolling Z-score for Corn Volume
logv = np.log1p(corn['corn_volume'])
corn['corn_volume_roll_z'] = (logv - logv.rolling(20).mean())/logv.rolling(20).std()

# Rolling RV for Corn
corn["corn_rv_5d"] = np.sqrt(corn["corn_ret"].rolling(5).apply(lambda x: np.sum(x**2), raw=True))
corn["corn_rv_20d"] = np.sqrt(corn["corn_ret"].rolling(20).apply(lambda x: np.sum(x**2), raw=True))

# Seasonality term
corn['month'] = corn.index.month

# 14 day Average True Range
high = corn.pop("corn_high")
low = corn.pop("corn_low")
close = corn["corn_close"]
prev_close = close.shift(1)
tr = pd.concat([high - low, (high - prev_close).abs(), (low - prev_close).abs()], axis=1).max(axis=1)
corn["atr_14"] = tr.rolling(14).mean()   # 14 is standard

# SMA Spread
sma_5 = corn["corn_close"].rolling(5).mean()
sma_20 = corn["corn_close"].rolling(20).mean()
corn["sma_spread"] = (sma_5 - sma_20) / sma_20

# MACD (12, 26, 9) - hozrizons are standard 
ema_12 = corn["corn_close"].ewm(span=12, adjust=False).mean()
ema_26 = corn["corn_close"].ewm(span=26, adjust=False).mean()
macd = ema_12 - ema_26
corn['macd_(12,26,9)'] = macd.ewm(span=9, adjust=False).mean()

# 14-day RSI
delta = corn["corn_close"].diff()
gain = delta.clip(lower=0)
loss = -delta.clip(upper=0)
avg_gain = gain.rolling(14).mean()
avg_loss = loss.rolling(14).mean()
rs = avg_gain / (avg_loss + 1e-10)  # avoid divide-by-zero error
corn["rsi_14"] = 100 - (100 / (1 + rs))

# Rolling Z-score for Oil and Soy Close
WINDOW = 20             # 20 day = ~3 week lookback window which is standard finance practice
MIN_PERIODS = round(0.75*WINDOW)
for name, cmdty in {"oil":oil, "soy":soy}.items():
    logp = np.log(cmdty[f"{name}_close"])
    cmdty[f"{name}_close_roll_z"] = (logp - logp.rolling(WINDOW, min_periods=MIN_PERIODS).mean())/logp.rolling(WINDOW, min_periods=MIN_PERIODS).std()

# Left join onto corn because we predict corn so that's the timeline that matters. Forward fill the rest of the parameters.
# Nulls are due to market open dates not mapping, oil, soy and corn data is complete otherwise...
cmdty_df = corn.join([oil, soy], how="left")
cmdty_df['corn_ret_t+1'] = cmdty_df['corn_ret'].shift(-1)       # Our Y is corn returns at t+1
cmdty_df = cmdty_df.loc['2000-10-01':'2026-04-09']              # Keep data from 2000-10-01 to '2026-04-09' since that's when all feature data is available latest
cmdty_df[["oil_close","oil_volume","soy_close","soy_volume"]] = cmdty_df[["oil_close","oil_volume","soy_close","soy_volume"]].ffill()

# Corn-Soy and Corn-Oil Ratio
cmdty_df["corn_soy_ratio"] = np.log(cmdty_df["corn_close"] / cmdty_df["soy_close"])
cmdty_df["corn_oil_ratio"] = np.log(cmdty_df["corn_close"] / cmdty_df["oil_close"])

# Yield Curve Features
xasset_df = cmdty_df.join(rates, how="left").ffill()
xasset_df['yc_10_2'] = xasset_df['T10_yield'] - xasset_df['T2_yield']
xasset_df['yc_2_3m'] = xasset_df['T2_yield'] - xasset_df['T3M_yield']
xasset_df['yc_curvature'] = 2*xasset_df['T5_yield'] - xasset_df['T2_yield'] - xasset_df['T10_yield']

# Interest Rate Change Features
xasset_df['d(yc_10_2)'] = xasset_df['yc_10_2'].diff(1)    # not pct change since yields are already rates
xasset_df['d(yc_2_3m)'] = xasset_df['yc_2_3m'].diff(1)
xasset_df['d(yc_curvature)'] = xasset_df['yc_curvature'].diff(1)
xasset_df['d(T10_yield)'] = xasset_df['T10_yield'].diff(1)
xasset_df['d(T2_yield)'] = xasset_df['T2_yield'].diff(1)

# Merge cross-asset features with weather features
wx_w_dly.rename(columns={"date":"Date"}, inplace=True)
xasset_df.reset_index(inplace=True)
df = xasset_df.merge(wx_w_dly, on=["Date"], how='left').set_index("Date")
# Creating lags (t-1, t-2) for key momentum indicators
lag_features = ['corn_rv_5d', 'rsi_14', 'd(yc_10_2)']
for col in lag_features:
    df[f'{col}_lag1'] = df[col].shift(1)
    df[f'{col}_lag2'] = df[col].shift(2)

df

/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)
/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/pandas/core/arraylike.py:399: RuntimeWarning: invalid value encountered in log
  result = getattr(ufunc, method)(*inputs, **kwargs)


,corn_close,corn_volume,corn_ret,corn_volume_roll_z,corn_rv_5d,corn_rv_20d,month,atr_14,sma_spread,"macd_(12,26,9)",...,prod_w_prcp,prod_w_rhum,prod_w_wspd,temp_prcp_interaction,corn_rv_5d_lag1,corn_rv_5d_lag2,rsi_14_lag1,rsi_14_lag2,d(yc_10_2)_lag1,d(yc_10_2)_lag2
Date,,,,,,,,,,,,,,,,,,,,,
2000-10-02,192.00,3.0,0.000000,-0.534580,0.021779,0.053082,10.0,2.107143,0.028748,2.094062,...,0.0,48.411058,9.724337,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2000-10-03,195.25,3.0,0.016927,-0.484432,0.018853,0.055290,10.0,1.857143,0.031072,2.314839,...,1.036692,51.591682,11.692289,14.212231,0.021779,NaN,68.674699,NaN,NaN,NaN
2000-10-04,199.50,3.0,0.021767,-0.520727,0.028676,0.059405,10.0,1.964286,0.036285,2.608881,...,5.597392,60.499126,13.658984,60.246159,0.018853,0.021779,70.786517,68.674699,2.000000e-02,NaN
2000-10-05,200.50,3.0,0.005013,-0.520727,0.029111,0.057702,10.0,2.535714,0.043004,2.943438,...,12.300664,67.352749,12.139667,88.277538,0.028676,0.018853,86.956522,70.786517,0.000000e+00,2.000000e-02
2000-10-06,199.50,3.0,-0.004988,-0.520727,0.028466,0.050595,10.0,2.785714,0.045840,3.263189,...,0.0,47.716438,13.418457,0.0,0.029111,0.028676,93.333333,86.956522,-2.000000e-02,0.000000e+00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2026-04-02,452.25,0.0,-0.004403,-2.807366,0.019868,0.050363,4.0,10.178571,0.001921,6.818618,...,0.490944,51.256284,10.833227,2.614031,0.019382,0.020559,55.309735,61.842105,1.000000e-02,-2.000000e-02
2026-04-06,454.00,126749.0,0.003870,0.403609,0.017178,0.048951,4.0,9.625000,-0.002358,6.369381,...,0.0,30.310008,9.399977,0.0,0.019868,0.019382,49.769585,55.309735,-4.440892e-16,1.000000e-02
2026-04-07,449.00,172653.0,-0.011013,0.429768,0.015276,0.045451,4.0,9.410714,-0.006572,5.821673,...,0.0,17.532417,5.801164,0.0,0.017178,0.019868,50.000000,49.769585,-2.000000e-02,-4.440892e-16


In [5]:
df.to_csv(path_or_buf="enriched_corn_data.csv", index=True, header=True)